# CLT-006: Control Loop Single and Double Exposure Test

Owner: **Bryce Kalmbach** <br>
Last Verified to Run: **2025-04-03** <br>
Software Version:
  - `ts_wep`: **14.1.1**
  - `donut_viz`: **1.6.2**
  - `lsst_distrib`: **w_2025_13**

## Test Details:
In this experiment we calculate the closed loop with 15 second exposure then reset the state back to the initial state and re-run the closed loop with 30 seconds.
We then compare the residual AOS FWHM during the progression of the closed loop.

In [ ]:
# Times Square Parameters
collection_name = 'u/brycek/aosRefitWcs_danish_singleBlends_80pxMinSep'
day_obs = 20241118
min_seq_num_15_sec = 31
max_seq_num_15_sec = 53
min_seq_num_30_sec = 54
max_seq_num_30_sec = 75

In [ ]:
import numpy as np
from copy import copy
from matplotlib import pyplot as plt
from lsst.daf.butler import Butler

# Can uncomment when running outside Times Square (Hopefully temporary)
#from lsst.ts.wep.utils import getPsfGradPerZernike

from astropy.table import Table
%matplotlib inline

In [ ]:
butler = Butler('/repo/main')

In [ ]:
camera = butler.get('camera', {'instrument': "LSSTComCam"}, collections=collection_name)

## Define Functions Needed (Temporary)
This is hopefully temporary since this function lives in `ts_wep` which is currently unavailable from Times Square

In [ ]:
import galsim

def getPsfGradPerZernike(
    diameter: float = 8.36,
    obscuration: float = 0.612,
    jmin: int = 4,
    jmax: int = 22,
) -> np.ndarray:
    """Get the gradient of the PSF FWHM with respect to each Zernike.

    This function takes no positional arguments. All parameters must be passed
    by name (see the list of parameters below).

    Parameters
    ----------
    diameter : float, optional
        The diameter of the telescope aperture, in meters.
        (the default, 8.36, corresponds to the LSST primary mirror)
    obscuration : float, optional
        Central obscuration of telescope aperture (i.e. R_outer / R_inner).
        (the default, 0.612, corresponds to the LSST primary mirror)
    jmin : int, optional
        The minimum Noll index, inclusive. Must be >= 0. (the default is 4)
    jmax : int, optional
        The max Zernike Noll index, inclusive. Must be >= jmin.
        (the default is 22.)

    Returns
    -------
    np.ndarray
        Gradient of the PSF FWHM with respect to the corresponding Zernike.
        Units are arcsec / micron.

    Raises
    ------
    ValueError
        If jmin is negative or jmax is less than jmin
    """
    # Check jmin and jmax
    if jmin < 0:
        raise ValueError("jmin cannot be negative.")
    if jmax < jmin:
        raise ValueError("jmax must be greater than jmin.")

    # Calculate the conversion factors
    conversion_factors = np.zeros(jmax + 1)
    for i in range(jmin, jmax + 1):
        # Set coefficients for this Noll index: coefs = [0, 0, ..., 1]
        # Note the first coefficient is Noll index 0, which does not exist and
        # is therefore always ignored by galsim
        coefs = [0] * i + [1]

        # Create the Zernike polynomial with these coefficients
        R_outer = diameter / 2
        R_inner = R_outer * obscuration
        Z = galsim.zernike.Zernike(coefs, R_outer=R_outer, R_inner=R_inner)

        # We can calculate the size of the PSF from the RMS of the gradient of
        # the wavefront. The gradient of the wavefront perturbs photon paths.
        # The RMS quantifies the size of the collective perturbation.
        # If we expand the wavefront gradient in another series of Zernike
        # polynomials, we can exploit the orthonormality of the Zernikes to
        # calculate the RMS from the Zernike coefficients.
        rms_tilt = np.sqrt(np.sum(Z.gradX.coef**2 + Z.gradY.coef**2) / 2)

        # Convert to arcsec per micron
        rms_tilt = np.rad2deg(rms_tilt * 1e-6) * 3600

        # Convert rms -> fwhm
        fwhm_tilt = 2 * np.sqrt(2 * np.log(2)) * rms_tilt

        # Save this conversion factor
        conversion_factors[i] = fwhm_tilt

    return conversion_factors[jmin:]


## Gather Visit Data

In [ ]:
visit_tables_15_sec = butler.query_datasets('aggregateAOSVisitTableAvg', 
                                               collections=collection_name,
                                               where=f"exposure.day_obs = {day_obs} and exposure.seq_num >= {min_seq_num_15_sec} and exposure.seq_num <= {max_seq_num_15_sec} and instrument = 'LSSTComCam'")
visit_tables_30_sec = butler.query_datasets('aggregateAOSVisitTableAvg', 
                                               collections=collection_name,
                                               where=f"exposure.day_obs = {day_obs} and exposure.seq_num >= {min_seq_num_30_sec} and exposure.seq_num <= {max_seq_num_30_sec} and instrument = 'LSSTComCam'")

In [ ]:
visit_dict_15_sec = dict()
for visit_ref in visit_tables_15_sec:
    visit_table = butler.get(visit_ref)
    visit_dict_15_sec[visit_table.meta['visit']] = visit_table
visit_dict_15_sec = dict(sorted(visit_dict_15_sec.items()))

In [ ]:
visit_dict_30_sec = dict()
for visit_ref in visit_tables_30_sec:
    visit_table = butler.get(visit_ref)
    visit_dict_30_sec[visit_table.meta['visit']] = visit_table
visit_dict_30_sec = dict(sorted(visit_dict_30_sec.items()))

In [ ]:
noll_idx = visit_table.meta['nollIndices']
noll_min = np.min(noll_idx)
noll_max = np.max(noll_idx)

In [ ]:
conv_array = getPsfGradPerZernike(jmin=noll_min, jmax=noll_max)

In [ ]:
zk_table_15_sec = Table()
zk_table_15_sec = Table(names=['visit', 'full_array', 'fwhm_combined'], dtype=[int, (float, 25), float])
for visit_id, visit_results in visit_dict_15_sec.items():
    zk_array = np.zeros((noll_max - noll_min + 1))
    for det_name in camera.getNameIter():
        zk_det_array = np.zeros((noll_max - noll_min + 1))
        zk_det_array[noll_idx - noll_min - 1] = visit_results[visit_results['detector'] == det_name]['zk_CCS']
        zk_array += zk_det_array * conv_array
    zk_array /= len(camera)
    zk_table_15_sec.add_row(vals=[visit_id, zk_array, np.sqrt(np.sum(zk_array**2))])

In [ ]:
zk_table_30_sec = Table()
zk_table_30_sec = Table(names=['visit', 'full_array', 'fwhm_combined'], dtype=[int, (float, 25), float])
for visit_id, visit_results in visit_dict_30_sec.items():
    zk_array = np.zeros((noll_max - noll_min + 1))
    for det_name in camera.getNameIter():
        zk_det_array = np.zeros((noll_max - noll_min + 1))
        zk_det_array[noll_idx - noll_min - 1] = visit_results[visit_results['detector'] == det_name]['zk_CCS']
        zk_array += zk_det_array * conv_array
    zk_array /= len(camera)
    zk_table_30_sec.add_row(vals=[visit_id, zk_array, np.sqrt(np.sum(zk_array**2))])

## Display results

In [ ]:
plt.plot((zk_table_15_sec['visit'] - zk_table_15_sec['visit'][0])/3, zk_table_15_sec['fwhm_combined'], label='15 Second Exposures')
plt.plot((zk_table_30_sec['visit'] - zk_table_30_sec['visit'][0])/3, zk_table_30_sec['fwhm_combined'], label='30 Second Exposures')
plt.xlabel('Closed Loop Iteration')
plt.ylabel('FWHM AOS Residual (arcsec)')
plt.legend()
band = visit_table.meta['band']
plt.title(
    'Closed Loop Convergence: 15 sec vs 30 sec.\n ' +
    f'day_obs: {day_obs} seq_num: {min_seq_num_15_sec} - {max_seq_num_15_sec}, {min_seq_num_30_sec} - {max_seq_num_30_sec}, band: {band}'
)